In [1]:
import pandas as pd
import numpy as np
import os

In [148]:
gem_met_info_f = "/home/users/gsantamaria/projects/metabo_pert/data/GEMs/Human2/Human-GEM/model/metabolites.tsv"
gem_cpd_f = "/home/users/gsantamaria/projects/metabo_pert/scripts/cpd_info.csv"
sch_cpd_f = "/home/users/gsantamaria/projects/metabo_pert/data/schuhknecht/NBT-RA62370A-Z_NonPooledFoldChanges.xlsx"
out_dir = "/home/users/gsantamaria/projects/metabo_pert/results/gem_cpd_mapping/"

os.makedirs(out_dir, exist_ok=True)

In [3]:
def is_nan(value):
    return (value != value)

In [4]:
gem_met_info = pd.read_csv(gem_met_info_f, sep = '\t', index_col = 0)

In [5]:
gem_cpd_info = pd.read_csv(gem_cpd_f, index_col = 0)

In [6]:
gem_met_info

,metsNoComp,metBiGGID,metKEGGID,metHMDBID,metChEBIID,metPubChemID,metLipidMapsID,metEHMNID,metHepatoNET1ID,metRecon3DID,metMetaNetXID,metHMR2ID,metRetired
mets,,,,,,,,,,,,,
MAM00001c,MAM00001,carveol,C00964,NaN,CHEBI:15389,NaN,NaN,NaN,NaN,carveol,MNXM45735,m00001c,m00001c
MAM00001e,MAM00001,carveol,C00964,NaN,CHEBI:15389,NaN,NaN,NaN,NaN,carveol,MNXM45735,m00001s,m00001s
MAM00002c,MAM00002,appnn,C09880,HMDB0006525,CHEBI:36740,6654.0,NaN,NaN,NaN,appnn,MNXM163755,m00002c,m00002c
MAM00002e,MAM00002,appnn,C09880,HMDB0006525,CHEBI:36740,6654.0,NaN,NaN,NaN,appnn,MNXM163755,m00002s,m00002s
MAM00003c,MAM00003,NaN,NaN,NaN,CHEBI:78990,NaN,LMFA01030283,NaN,NaN,M00003,MNXM150165;MNXM27815,m00003c,m00003c
...,...,...,...,...,...,...,...,...,...,...,...,...,...
MAM00077c,MAM00077,NaN,CE2416,HMDB0000795,CHEBI:51340,123929.0,LMPR0104010022,NaN,NaN,NaN,MNXM747404,NaN,NaN
MAM00077e,MAM00077,NaN,CE2416,HMDB0000795,CHEBI:51340,123929.0,LMPR0104010022,NaN,NaN,NaN,MNXM747404,NaN,NaN
MAM20084m,MAM20084,4fe4s,NaN,NaN,CHEBI:33722,6398953.0,NaN,NaN,NaN,NaN,MNXM732007,NaN,NaN


In [7]:
sch_xlsx = pd.ExcelFile(sch_cpd_f)
sch_cpd = sch_xlsx.parse("Metainformation_Ions")

In [8]:
sch_cpd

,Annotation,Ion m/z,Annotation IDs,Ionization
0,Acetone,57.033752,C00207,neg
1,Guanidine,58.038200,NaN,neg
2,Glycolaldehyde ; Acetate,59.013775,C00266 ; C00033,neg
3,Urea,59.027712,C00086,neg
4,Isopropyl alcohol,59.048298,HMDB0000863,neg
...,...,...,...,...
2264,"TG(22:4(7Z,10Z,13Z,16Z)/18:2(9Z,12Z)/22:5(7Z,1...",993.791504,HMDB0054781,neg
2265,"2,4,7,10,13-Hexadecapentaenoyl Coenzyme A",994.259766,NaN,neg
2266,"Trans,Cis-Hexadeca-2,7-Dienoyl Coenzyme A ; 2,...",996.275269,NaN,neg
2267,"7Z-Hexadecenoyl Coenzyme A ; 2,7,10-Hexadecatr...",998.291016,NaN,neg


In [9]:
match_column = []
matchid_column = []
for i in range(0, gem_met_info.shape[0]):
    kegg = gem_met_info["metKEGGID"][i]
    hmdb = gem_met_info["metHMDBID"][i]
    if i % 500 == 0:
        print("Compound %i/%i" %(i, gem_met_info.shape[0]))
    matches = []
    matching_ids = []
    for j in range(0, sch_cpd.shape[0]):
        if not is_nan(sch_cpd["Annotation IDs"][j]):
            cpd_id = sch_cpd["Annotation IDs"][j].split(' ; ')
            if kegg in cpd_id or hmdb in cpd_id:
                matches.append(sch_cpd["Annotation"][j])
                if kegg in cpd_id:
                    matching_ids.append(kegg)
                if hmdb in cpd_id:
                    matching_ids.append(hmdb)
    if len(matches) > 0:
        matches = '|'.join(np.unique(matches))
    else:
        matches = None
    if len(matching_ids) > 0:
        matching_ids = '|'.join(np.unique(matching_ids))
    else:
        matching_ids = None
    match_column.append(matches)
    matchid_column.append(matching_ids)

Compound 0/8461


/tmp/ipykernel_614415/161534252.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  kegg = gem_met_info["metKEGGID"][i]
/tmp/ipykernel_614415/161534252.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  hmdb = gem_met_info["metHMDBID"][i]


Compound 500/8461
Compound 1000/8461
Compound 1500/8461
Compound 2000/8461
Compound 2500/8461
Compound 3000/8461
Compound 3500/8461
Compound 4000/8461
Compound 4500/8461
Compound 5000/8461
Compound 5500/8461
Compound 6000/8461
Compound 6500/8461
Compound 7000/8461
Compound 7500/8461
Compound 8000/8461


In [10]:
gem_met_info["Schuhknecht_mapping"] = match_column
gem_met_info["id_mapped_to_Schuhknecht"] = matchid_column

In [11]:
gem_met_info

,metsNoComp,metBiGGID,metKEGGID,metHMDBID,metChEBIID,metPubChemID,metLipidMapsID,metEHMNID,metHepatoNET1ID,metRecon3DID,metMetaNetXID,metHMR2ID,metRetired,Schuhknecht_mapping,id_mapped_to_Schuhknecht
mets,,,,,,,,,,,,,,,
MAM00001c,MAM00001,carveol,C00964,NaN,CHEBI:15389,NaN,NaN,NaN,NaN,carveol,MNXM45735,m00001c,m00001c,(-)-Trans-Carveol ; Alpha-Pinene Oxide ; Apoca...,C00964
MAM00001e,MAM00001,carveol,C00964,NaN,CHEBI:15389,NaN,NaN,NaN,NaN,carveol,MNXM45735,m00001s,m00001s,(-)-Trans-Carveol ; Alpha-Pinene Oxide ; Apoca...,C00964
MAM00002c,MAM00002,appnn,C09880,HMDB0006525,CHEBI:36740,6654.0,NaN,NaN,NaN,appnn,MNXM163755,m00002c,m00002c,(+)-Alpha-Pinene ; (+)-a-Pinene ; (-)-alpha-Pi...,C09880|HMDB0006525
MAM00002e,MAM00002,appnn,C09880,HMDB0006525,CHEBI:36740,6654.0,NaN,NaN,NaN,appnn,MNXM163755,m00002s,m00002s,(+)-Alpha-Pinene ; (+)-a-Pinene ; (-)-alpha-Pi...,C09880|HMDB0006525
MAM00003c,MAM00003,NaN,NaN,NaN,CHEBI:78990,NaN,LMFA01030283,NaN,NaN,M00003,MNXM150165;MNXM27815,m00003c,m00003c,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MAM00077c,MAM00077,NaN,CE2416,HMDB0000795,CHEBI:51340,123929.0,LMPR0104010022,NaN,NaN,NaN,MNXM747404,NaN,NaN,Methyl stearate ; Nonadecanoic acid ; Pristani...,HMDB0000795
MAM00077e,MAM00077,NaN,CE2416,HMDB0000795,CHEBI:51340,123929.0,LMPR0104010022,NaN,NaN,NaN,MNXM747404,NaN,NaN,Methyl stearate ; Nonadecanoic acid ; Pristani...,HMDB0000795
MAM20084m,MAM20084,4fe4s,NaN,NaN,CHEBI:33722,6398953.0,NaN,NaN,NaN,NaN,MNXM732007,NaN,NaN,None,None


In [12]:
gem_met_info

,metsNoComp,metBiGGID,metKEGGID,metHMDBID,metChEBIID,metPubChemID,metLipidMapsID,metEHMNID,metHepatoNET1ID,metRecon3DID,metMetaNetXID,metHMR2ID,metRetired,Schuhknecht_mapping,id_mapped_to_Schuhknecht
mets,,,,,,,,,,,,,,,
MAM00001c,MAM00001,carveol,C00964,NaN,CHEBI:15389,NaN,NaN,NaN,NaN,carveol,MNXM45735,m00001c,m00001c,(-)-Trans-Carveol ; Alpha-Pinene Oxide ; Apoca...,C00964
MAM00001e,MAM00001,carveol,C00964,NaN,CHEBI:15389,NaN,NaN,NaN,NaN,carveol,MNXM45735,m00001s,m00001s,(-)-Trans-Carveol ; Alpha-Pinene Oxide ; Apoca...,C00964
MAM00002c,MAM00002,appnn,C09880,HMDB0006525,CHEBI:36740,6654.0,NaN,NaN,NaN,appnn,MNXM163755,m00002c,m00002c,(+)-Alpha-Pinene ; (+)-a-Pinene ; (-)-alpha-Pi...,C09880|HMDB0006525
MAM00002e,MAM00002,appnn,C09880,HMDB0006525,CHEBI:36740,6654.0,NaN,NaN,NaN,appnn,MNXM163755,m00002s,m00002s,(+)-Alpha-Pinene ; (+)-a-Pinene ; (-)-alpha-Pi...,C09880|HMDB0006525
MAM00003c,MAM00003,NaN,NaN,NaN,CHEBI:78990,NaN,LMFA01030283,NaN,NaN,M00003,MNXM150165;MNXM27815,m00003c,m00003c,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MAM00077c,MAM00077,NaN,CE2416,HMDB0000795,CHEBI:51340,123929.0,LMPR0104010022,NaN,NaN,NaN,MNXM747404,NaN,NaN,Methyl stearate ; Nonadecanoic acid ; Pristani...,HMDB0000795
MAM00077e,MAM00077,NaN,CE2416,HMDB0000795,CHEBI:51340,123929.0,LMPR0104010022,NaN,NaN,NaN,MNXM747404,NaN,NaN,Methyl stearate ; Nonadecanoic acid ; Pristani...,HMDB0000795
MAM20084m,MAM20084,4fe4s,NaN,NaN,CHEBI:33722,6398953.0,NaN,NaN,NaN,NaN,MNXM732007,NaN,NaN,None,None


In [13]:
gem_cpd_info

,id,name,formula,compartment,reactions,subsystems
0,MAM00001c,(-)-trans-carveol,C10H16O,c,MAR08597|MAR08596,Transport reactions|Xenobiotics metabolism
1,MAM00001e,(-)-trans-carveol,C10H16O,e,MAR08597|MAR09277,Transport reactions|Exchange/demand reactions
2,MAM00002c,(+)-alpha-pinene,C10H16,c,MAR08601|MAR08600,Xenobiotics metabolism|Transport reactions
3,MAM00002e,(+)-alpha-pinene,C10H16,e,MAR08600|MAR09267,Transport reactions|Exchange/demand reactions
4,MAM00003c,(10Z)-heptadecenoic acid,C17H31O2,c,MAR00240|MAR00686|MAR00243|MAR00691|MAR10033|M...,Transport reactions|Pool reactions|Acyl-CoA hy...
...,...,...,...,...,...,...
8456,MAM20082m,S-propanoyldihydrolipoamide,C11H23N1O2S2,m,MAR20178|MAR20177,Propanoate metabolism
8457,MAM20083x,(2R)-pristanal,C19H38O,x,MAR03488|MAR03486,Beta oxidation of phytanic acid (peroxisomal)
8458,MAM20084m,4Fe4S iron-sulfur cluster,NaN,m,MAR20184|MAR06403,Sulfur metabolism|Lipoic acid metabolism
8459,MAM20085m,2Fe2S iron-sulfur cluster,NaN,m,MAR20185|MAR20184,Sulfur metabolism


In [14]:
# Add columns of interest to gem_cpd_info

columns = [
    "metKEGGID",
    "metHMDBID",
    "Schuhknecht_mapping",
    "id_mapped_to_Schuhknecht"
]

gem_cpd_info[columns] = gem_met_info.loc[
    gem_cpd_info["id"], columns
].to_numpy()

In [15]:
# Get a version with unique compounds

columns = [
    "name",
    "formula",
    "metKEGGID",
    "metHMDBID",
    "Schuhknecht_mapping",
    "id_mapped_to_Schuhknecht"
]

gem_cpd_info_uniq = gem_cpd_info[columns].drop_duplicates().reset_index(drop=True)

In [16]:
# Keep only the ones with mapped Schuhknecht id
gem_cpd_info_uniq = gem_cpd_info_uniq.loc[
    [x != None for x in gem_cpd_info_uniq["Schuhknecht_mapping"]], :
]

In [17]:
gem_cpd_info_uniq.shape

(919, 6)

In [18]:
gem_cpd_info_uniq["name"]

0                                       (-)-trans-carveol
1                                        (+)-alpha-pinene
7                            (11Z,14Z)-eicosadienoic acid
9                       (11Z,14Z,17Z)-eicosatrienoic acid
20                           (13Z,16Z)-docosadienoic acid
                              ...                        
3883                                   (3S)-phytanic acid
3922                                          Sebacicacid
3932                                   S-Sulfo-L-Cysteine
3953                                         Suberic Acid
3993    Taurohyocholic acid; N-(3alpha,6alpha,7alpha-T...
Name: name, Length: 919, dtype: object

In [19]:
# Remove IDs with duplicated names, after verifying why is that: in the metabolite
# info of the gem some rows of different compartments have HMDB IDs while others don't

gem_name_dups = gem_cpd_info_uniq[gem_cpd_info_uniq["name"].duplicated(keep=False)]
gem_name_dups

,name,formula,metKEGGID,metHMDBID,Schuhknecht_mapping,id_mapped_to_Schuhknecht
1123,5-oxoproline,C5H6NO3,C01879,HMDB0000267,5-L-Gamma-Glutamyl Amino Acid ; PL_Dimethadion...,C01879
1124,5-oxoproline,C5H6NO3,C01879,NaN,5-L-Gamma-Glutamyl Amino Acid ; PL_Dimethadion...,C01879
1778,elaidate,C18H33O2,C00712,HMDB0000573,Elaidic acid ; Oleic acid ; Vaccenic acid ; (9...,C00712|HMDB0000573
1779,elaidate,C18H33O2,C00712,NaN,Elaidic acid ; Oleic acid ; Vaccenic acid ; (9...,C00712
1913,galactose,C6H12O6,C00984,HMDB0000143,(Gal)2-Xyl-L-Ser (Protein) ; D-Fructose ; D-Ga...,C00984
1914,galactose,C6H12O6,C00984,NaN,(Gal)2-Xyl-L-Ser (Protein) ; D-Fructose ; D-Ga...,C00984


In [20]:
gem_name_dups["name"]

1123    5-oxoproline
1124    5-oxoproline
1778        elaidate
1779        elaidate
1913       galactose
1914       galactose
Name: name, dtype: object

In [21]:
gem_cpd_info_uniq.shape

(919, 6)

In [22]:
non_dups_to_keep = [
    not(n in gem_name_dups["name"].to_list() and is_nan(h))
    for n, h in zip(
        gem_cpd_info_uniq["name"], gem_cpd_info_uniq["metHMDBID"]
    )
]

gem_cpd_info_uniq = gem_cpd_info_uniq.loc[non_dups_to_keep, :]

In [23]:
gem_cpd_info_uniq.shape

(916, 6)

In [46]:
# Check what GEM compounds map to multiple Schuhknecht names/IDs by looking
# for the '|' character in their names

#pd.set_option("display.max_colwidth", None)

gem_cpd_info_uniq_map_to_multiple_shuhk = gem_cpd_info_uniq.loc[
    [
        '|' in x for x in gem_cpd_info_uniq["Schuhknecht_mapping"]
    ], :
]#[["name", "Schuhknecht_mapping"]]

In [47]:
gem_cpd_info_uniq_map_to_multiple_shuhk

,name,formula,metKEGGID,metHMDBID,Schuhknecht_mapping,id_mapped_to_Schuhknecht
7,"(11Z,14Z)-eicosadienoic acid",C20H35O2,C16525,NaN,"11Z,14Z-Eicosadienoic Acid ; 8Z,11Z-Eicosadien...",C16525
334,13(S)-HPODE,C18H31O4,C04717,HMDB0003871,12(13)Ep-9-KODE ; 13-L-Hydroperoxylinoleic Aci...,C04717|HMDB0003871
338,"13,16,19-docosatrienoic acid",C22H37O2,C16534,HMDB0002823,"(Gal)4 (Glc)1 (GlcNAc)3 (LFuc)2 (Cer)1 ; 10Z,1...",C16534|HMDB0002823
653,2-lysolecithin pool,C8H19NO5PRCO2,C04230,NaN,1-Eicosatetraenoylglycerophosphocholine (Delta...,C04230
668,2-oxobutyrate,C4H5O3,C00109,HMDB0000005,PHderiv_2-Oxobutanoate; 2-Ketobutyric acid; 2-...,C00109
669,2-oxoglutaramate,C5H6NO4,C00940,NaN,2-Oxoglutaramate|PHderiv_2-Oxoglutaramate; 2-O...,C00940
820,3-methyl-2-oxobutyrate,C5H7O3,C00141,HMDB0000019,2-Methylacetoacetic acid ; Alpha-ketoisovaleri...,C00141|HMDB0000019
915,3-sulfopyruvate,C3H2O6S,C05528,HMDB0004045,3-Sulfonatopyruvate|PHderiv_3-Sulfopyruvate; 3...,C05528
1008,4-methyl-2-oxopentanoate,C6H9O3,C00233,HMDB0000695,2-Methyl-3-ketovaleric acid ; 3-Methyl-2-oxova...,C00233|HMDB0000695
1069,5-amino-2-oxopentanoic acid,C5H9NO3,C01110,HMDB0006272,5-Amino-2-Oxopentanoic Acid ; 5-Amino-4-Oxopen...,C01110


In [50]:
for i in range(0, gem_cpd_info_uniq_map_to_multiple_shuhk.shape[0]):
    print(
        " *** %s - %s" % (
            gem_cpd_info_uniq_map_to_multiple_shuhk["name"].iloc[i],
            gem_cpd_info_uniq_map_to_multiple_shuhk["Schuhknecht_mapping"].iloc[i]
        )
    )

 *** (11Z,14Z)-eicosadienoic acid - 11Z,14Z-Eicosadienoic Acid ; 8Z,11Z-Eicosadienoic Acid|Eicosadienoic acid ; Cis,Cis-11,14-Eicosadienoic Acid
 *** 13(S)-HPODE - 12(13)Ep-9-KODE ; 13-L-Hydroperoxylinoleic Acid|12,13-DiHODE ; 13-L-Hydroperoxylinoleic acid ; 15,16-DiHODE ; 9(S)-HPODE ; 9,10-DiHODE
 *** 13,16,19-docosatrienoic acid - (Gal)4 (Glc)1 (GlcNAc)3 (LFuc)2 (Cer)1 ; 10Z,13Z,16Z-Docosatriynoic Acid ; 13Z,16Z,19Z-Docosatrienoic Acid ; Gm2A ; Lactoisooctaosylceramide ((Gal)4 (Glc)1 (GlcNAc)3 (Cer)1) ; Monofucosyllactoisooctaosylceramide ; Monosialylgalactosylgloboside ; V3(Neuac)2-Gb5Cer ((Gal)3 (GalNAc)1 (Glc)1 (Neu5Ac)2 (Cer)1)|Docosatrienoic acid
 *** 2-lysolecithin pool - 1-Eicosatetraenoylglycerophosphocholine (Delta 8, 11, 14, 17), Sn1-Lpc (20:4)|1-Heptadecanoylglycerophosphocholine ; LysoPE(20:0/0:0) ; LysoPC(17:0)|1-Lignocericylglycerophosphocholine (24:0), Lysopc A C24|1-Myristoylglycerophosphocholine ; 1-Heptadecanoylglycerophosphoethanolamine (C17:0 Pe) ; LysoPC(14:0)|1-

In [120]:
print(
    " *** %s - %s" % (
        gem_cpd_info_uniq_map_to_multiple_shuhk["name"].iloc[28],
        gem_cpd_info_uniq_map_to_multiple_shuhk["Schuhknecht_mapping"].iloc[28]
    )
)

 *** 1-Oleoylglycerophosphocholine (Delta 9) - 1-Eicosatetraenoylglycerophosphocholine (Delta 8, 11, 14, 17), Sn1-Lpc (20:4)|1-Heptadecanoylglycerophosphocholine ; LysoPE(20:0/0:0) ; LysoPC(17:0)|1-Lignocericylglycerophosphocholine (24:0), Lysopc A C24|1-Myristoylglycerophosphocholine ; 1-Heptadecanoylglycerophosphoethanolamine (C17:0 Pe) ; LysoPC(14:0)|1-Oleoylglycerophosphocholine (Delta 9) ; 2-Oleoylglycerophosphocholine ; LysoPC(18:1(11Z)) ; LysoPC(18:1(9Z))|1-Stearoylglycerophosphocholine ; LysoPC(18:0)


In [121]:
# Determine manually for multiple matches the best match
#                [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29
best_match_idx = [0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 3, 1, 2, 4, 5]
match_is_ambig = [1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]

In [122]:
for i in range(0, gem_cpd_info_uniq_map_to_multiple_shuhk.shape[0]):
    print(gem_cpd_info_uniq_map_to_multiple_shuhk["Schuhknecht_mapping"].iloc[i].split('|')[best_match_idx[i]])

11Z,14Z-Eicosadienoic Acid ; 8Z,11Z-Eicosadienoic Acid
12(13)Ep-9-KODE ; 13-L-Hydroperoxylinoleic Acid
Docosatrienoic acid
1-Eicosatetraenoylglycerophosphocholine (Delta 8, 11, 14, 17), Sn1-Lpc (20:4)
PHderiv_2-Oxobutanoate; 2-Ketobutyric acid; 2-Oxobutyric acid; 2-Oxobutyrate; 2-Oxobutanoic acid; alpha-Ketobutyric acid; alpha-Ketobutyrate
PHderiv_2-Oxoglutaramate; 2-Oxoglutaramic acid; alpha-Ketoglutaramate; 2-Ketoglutaramate
PHderiv_3-Methyl-2-oxobutanoic acid; 3-Methyl-2-oxobutyric acid; 3-Methyl-2-oxobutanoate; 2-Oxo-3-methylbutanoate; 2-Oxoisovalerate; 2-Oxoisopentanoate; alpha-Ketovaline; 2-Ketovaline; 2-Keto-3-methylbutyric acid
PHderiv_3-Sulfopyruvate; 3-Sulfopyruvic acid
PHderiv_(S)-3-Methyl-2-oxopentanoic acid; (S)-3-Methyl-2-oxopentanoate; (3S)-3-Methyl-2-oxopentanoic acid; (3S)-3-Methyl-2-oxopentanoate ; PHderiv_2-Oxohexanoic acid; 2-Oxohexanoate; alpha-Ketocaproic acid ; PHderiv_4-Methyl-2-oxopentanoate; 2-Oxoisocaproate
PHderiv_5-Amino-2-oxopentanoic acid; 5-Amino-2-oxope

In [123]:
gem_cpd_info_uniq_map_to_multiple_shuhk = gem_cpd_info_uniq_map_to_multiple_shuhk.copy()

gem_cpd_info_uniq_map_to_multiple_shuhk["Schuhknecht_mapping_resolved"] = [
    gem_cpd_info_uniq_map_to_multiple_shuhk["Schuhknecht_mapping"].iloc[i].split('|')[best_match_idx[i]]
    for i in range(len(gem_cpd_info_uniq_map_to_multiple_shuhk))
]

In [125]:
gem_cpd_info_uniq_map_to_multiple_shuhk["Schuhknecht_ambig"] = match_is_ambig

In [126]:
gem_cpd_info_uniq_map_to_multiple_shuhk

,name,formula,metKEGGID,metHMDBID,Schuhknecht_mapping,id_mapped_to_Schuhknecht,Schuhknecht_mapping_resolved,Schuhknecht_ambig
7,"(11Z,14Z)-eicosadienoic acid",C20H35O2,C16525,NaN,"11Z,14Z-Eicosadienoic Acid ; 8Z,11Z-Eicosadien...",C16525,"11Z,14Z-Eicosadienoic Acid ; 8Z,11Z-Eicosadien...",1
334,13(S)-HPODE,C18H31O4,C04717,HMDB0003871,12(13)Ep-9-KODE ; 13-L-Hydroperoxylinoleic Aci...,C04717|HMDB0003871,12(13)Ep-9-KODE ; 13-L-Hydroperoxylinoleic Acid,1
338,"13,16,19-docosatrienoic acid",C22H37O2,C16534,HMDB0002823,"(Gal)4 (Glc)1 (GlcNAc)3 (LFuc)2 (Cer)1 ; 10Z,1...",C16534|HMDB0002823,Docosatrienoic acid,0
653,2-lysolecithin pool,C8H19NO5PRCO2,C04230,NaN,1-Eicosatetraenoylglycerophosphocholine (Delta...,C04230,1-Eicosatetraenoylglycerophosphocholine (Delta...,0
668,2-oxobutyrate,C4H5O3,C00109,HMDB0000005,PHderiv_2-Oxobutanoate; 2-Ketobutyric acid; 2-...,C00109,PHderiv_2-Oxobutanoate; 2-Ketobutyric acid; 2-...,0
669,2-oxoglutaramate,C5H6NO4,C00940,NaN,2-Oxoglutaramate|PHderiv_2-Oxoglutaramate; 2-O...,C00940,PHderiv_2-Oxoglutaramate; 2-Oxoglutaramic acid...,0
820,3-methyl-2-oxobutyrate,C5H7O3,C00141,HMDB0000019,2-Methylacetoacetic acid ; Alpha-ketoisovaleri...,C00141|HMDB0000019,PHderiv_3-Methyl-2-oxobutanoic acid; 3-Methyl-...,0
915,3-sulfopyruvate,C3H2O6S,C05528,HMDB0004045,3-Sulfonatopyruvate|PHderiv_3-Sulfopyruvate; 3...,C05528,PHderiv_3-Sulfopyruvate; 3-Sulfopyruvic acid,0
1008,4-methyl-2-oxopentanoate,C6H9O3,C00233,HMDB0000695,2-Methyl-3-ketovaleric acid ; 3-Methyl-2-oxova...,C00233|HMDB0000695,PHderiv_(S)-3-Methyl-2-oxopentanoic acid; (S)-...,1
1069,5-amino-2-oxopentanoic acid,C5H9NO3,C01110,HMDB0006272,5-Amino-2-Oxopentanoic Acid ; 5-Amino-4-Oxopen...,C01110,PHderiv_5-Amino-2-oxopentanoic acid; 5-Amino-2...,0


In [133]:
# Substutute the resolved compounds in the unique dataset

resolved = gem_cpd_info_uniq_map_to_multiple_shuhk.set_index("name")[
    "Schuhknecht_mapping_resolved"
]

gem_cpd_info_uniq["Schuhknecht_mapping"] = (
    gem_cpd_info_uniq["name"].map(resolved)
    .fillna(gem_cpd_info_uniq["Schuhknecht_mapping"])
)

In [132]:
resolved

name
(11Z,14Z)-eicosadienoic acid                                                     11Z,14Z-Eicosadienoic Acid ; 8Z,11Z-Eicosadien...
13(S)-HPODE                                                                        12(13)Ep-9-KODE ; 13-L-Hydroperoxylinoleic Acid
13,16,19-docosatrienoic acid                                                                                   Docosatrienoic acid
2-lysolecithin pool                                                              1-Eicosatetraenoylglycerophosphocholine (Delta...
2-oxobutyrate                                                                    PHderiv_2-Oxobutanoate; 2-Ketobutyric acid; 2-...
2-oxoglutaramate                                                                 PHderiv_2-Oxoglutaramate; 2-Oxoglutaramic acid...
3-methyl-2-oxobutyrate                                                           PHderiv_3-Methyl-2-oxobutanoic acid; 3-Methyl-...
3-sulfopyruvate                                                               

In [139]:
# Check what compounds, after being mapped to Schuhknecht, are duplicated
# in this collumn, to evaluate ambiguousness.

dup_shuhk = gem_cpd_info_uniq.loc[gem_cpd_info_uniq["Schuhknecht_mapping"].duplicated(keep=False), :]

In [144]:
gem_cpd_info_uniq["dup_after_shuhknecht_map"] = [x in dup_shuhk["Schuhknecht_mapping"].to_list() for x in gem_cpd_info_uniq["Schuhknecht_mapping"]]

In [149]:
gem_cpd_info_uniq.to_csv(os.path.join(out_dir, "gem_to_schuhknecht_unique_map.csv"))

In [151]:
gem_cpd_info_uniq

,name,formula,metKEGGID,metHMDBID,Schuhknecht_mapping,id_mapped_to_Schuhknecht,dup_after_shuhknecht_map
0,(-)-trans-carveol,C10H16O,C00964,NaN,(-)-Trans-Carveol ; Alpha-Pinene Oxide ; Apoca...,C00964,True
1,(+)-alpha-pinene,C10H16,C09880,HMDB0006525,(+)-Alpha-Pinene ; (+)-a-Pinene ; (-)-alpha-Pi...,C09880|HMDB0006525,True
7,"(11Z,14Z)-eicosadienoic acid",C20H35O2,C16525,NaN,"11Z,14Z-Eicosadienoic Acid ; 8Z,11Z-Eicosadien...",C16525,False
9,"(11Z,14Z,17Z)-eicosatrienoic acid",C20H33O2,C16522,NaN,"11Z,14Z,17Z-Eicosatrienoic Acid",C16522,False
20,"(13Z,16Z)-docosadienoic acid",C22H39O2,C16533,NaN,"13Z,16Z-Docosadienoic Acid",C16533,False
...,...,...,...,...,...,...,...
3883,(3S)-phytanic acid,C20H39O2,C01607,HMDB0000801,"3D,7D,11D-Phytanic acid ; 3L,7D,11D-Phytanic a...",C01607|HMDB0000801,True
3922,Sebacicacid,C10H16O4,C08277,HMDB0000792,Sebacic acid ; Sebacicacid,C08277|HMDB0000792,False
3932,S-Sulfo-L-Cysteine,C3H8NO5S2,C05824,HMDB0000731,S-Sulfo-L-Cysteine,C05824,False
3953,Suberic Acid,C8H12O4,C08278,HMDB0000893,Suberic Acid,C08278,False


In [152]:
# Add the Schuhknecht columns to the gem_cpd_info dataset

gem_cpd_info

# Substutute the resolved compounds in the unique dataset

mapping = gem_cpd_info_uniq.set_index("name")[
    "Schuhknecht_mapping"
]

gem_cpd_info["Schuhknecht_mapping"] = (
    gem_cpd_info["name"].map(mapping)
    .fillna(gem_cpd_info["Schuhknecht_mapping"])
)

In [153]:
gem_cpd_info

,id,name,formula,compartment,reactions,subsystems,metKEGGID,metHMDBID,Schuhknecht_mapping,id_mapped_to_Schuhknecht
0,MAM00001c,(-)-trans-carveol,C10H16O,c,MAR08597|MAR08596,Transport reactions|Xenobiotics metabolism,C00964,NaN,(-)-Trans-Carveol ; Alpha-Pinene Oxide ; Apoca...,C00964
1,MAM00001e,(-)-trans-carveol,C10H16O,e,MAR08597|MAR09277,Transport reactions|Exchange/demand reactions,C00964,NaN,(-)-Trans-Carveol ; Alpha-Pinene Oxide ; Apoca...,C00964
2,MAM00002c,(+)-alpha-pinene,C10H16,c,MAR08601|MAR08600,Xenobiotics metabolism|Transport reactions,C09880,HMDB0006525,(+)-Alpha-Pinene ; (+)-a-Pinene ; (-)-alpha-Pi...,C09880|HMDB0006525
3,MAM00002e,(+)-alpha-pinene,C10H16,e,MAR08600|MAR09267,Transport reactions|Exchange/demand reactions,C09880,HMDB0006525,(+)-Alpha-Pinene ; (+)-a-Pinene ; (-)-alpha-Pi...,C09880|HMDB0006525
4,MAM00003c,(10Z)-heptadecenoic acid,C17H31O2,c,MAR00240|MAR00686|MAR00243|MAR00691|MAR10033|M...,Transport reactions|Pool reactions|Acyl-CoA hy...,NaN,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...
8456,MAM20082m,S-propanoyldihydrolipoamide,C11H23N1O2S2,m,MAR20178|MAR20177,Propanoate metabolism,C21018,NaN,None,None
8457,MAM20083x,(2R)-pristanal,C19H38O,x,MAR03488|MAR03486,Beta oxidation of phytanic acid (peroxisomal),NaN,HMDB0001958,None,None
8458,MAM20084m,4Fe4S iron-sulfur cluster,NaN,m,MAR20184|MAR06403,Sulfur metabolism|Lipoic acid metabolism,NaN,NaN,None,None
8459,MAM20085m,2Fe2S iron-sulfur cluster,NaN,m,MAR20185|MAR20184,Sulfur metabolism,NaN,NaN,None,None


In [160]:
np.unique(gem_cpd_info["compartment"].to_list())

array(['c', 'e', 'g', 'i', 'l', 'm', 'n', 'r', 'x'], dtype='<U1')

In [157]:
mapping = gem_cpd_info_uniq.set_index("name")[
    "dup_after_shuhknecht_map"
]

gem_cpd_info["dup_after_shuhknecht_map"] = (
    gem_cpd_info["name"].map(mapping)
)

In [159]:
gem_cpd_info.to_csv(os.path.join(out_dir, "cpd_info_wSchuhknecht.csv"))